<a href="https://colab.research.google.com/github/vibhorjoshi/-CHECK/blob/main/Copy_of_scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semiconductor Image Restoration: TPU Training Pipeline (150 Epochs)
This notebook contains the optimized 12-block RRDB architecture and Hybrid Loss function (Charbonnier + SSIM + Gradient) migrated for TPU execution.

In [31]:
model_script = """import torch
import torch.nn as nn
import torch.nn.functional as F

class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super(RRDB, self).__init__()
        self.c1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.c2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.c3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.c4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.c5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
    def forward(self, x):
        x1 = F.leaky_relu(self.c1(x), 0.2, inplace=True)
        x2 = F.leaky_relu(self.c2(torch.cat((x, x1), 1)), 0.2, inplace=True)
        x3 = F.leaky_relu(self.c3(torch.cat((x, x1, x2), 1)), 0.2, inplace=True)
        x4 = F.leaky_relu(self.c4(torch.cat((x, x1, x2, x3), 1)), 0.2, inplace=True)
        x5 = self.c5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RestorationNet(nn.Module):
    def __init__(self, in_nc=1, out_nc=1, nf=64, nb=12, gc=32):
        super(RestorationNet, self).__init__()
        self.conv_first = nn.Conv2d(in_nc, nf, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_hr = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, out_nc, 3, 1, 1)
    def forward(self, x):
        fea = self.conv_first(x)
        body_fea = self.conv_body(self.body(fea))
        fea = fea + body_fea
        fea = F.leaky_relu(self.conv_up1(F.interpolate(fea, scale_factor=2, mode='nearest')), 0.2, inplace=True)
        fea = F.leaky_relu(self.conv_hr(fea), 0.2, inplace=True)
        out = self.conv_last(fea)
        return out
"""
with open('model.py', 'w') as f: f.write(model_script)

loss_script = """import torch
import torch.nn as nn
import torch.nn.functional as F

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3): super().__init__(); self.eps = eps
    def forward(self, x, y): return torch.mean(torch.sqrt((x - y)**2 + self.eps**2))

class GradientLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, x, y):
        h_x, w_x = x.size()[-2:], y.size()[-2:]
        return F.l1_loss(x[:,:,1:,:] - x[:,:,:-1,:], y[:,:,1:,:] - y[:,:,:-1,:]) + \
               F.l1_loss(x[:,:,:,1:] - x[:,:,:,:-1], y[:,:,:,1:] - y[:,:,:,:-1])

class CombinedLoss(nn.Module):
    def __init__(self): super().__init__(); self.cb = CharbonnierLoss(); self.grad = GradientLoss()
    def forward(self, pred, target): return self.cb(pred, target) + 0.1 * self.grad(pred, target)
"""
with open('losses.py', 'w') as f: f.write(loss_script)
print("Recreated model.py and losses.py.")

Recreated model.py and losses.py.


In [32]:
dataset_script = """import torch
import numpy as np
import glob
import os
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, degraded_path, gt_path):
        self.degraded_files = sorted(glob.glob(os.path.join(degraded_path, '*.npy')))
        self.gt_files = sorted(glob.glob(os.path.join(gt_path, '*.npy')))
    def __len__(self):
        return min(len(self.degraded_files), len(self.gt_files))
    def __getitem__(self, idx):
        x_arr = np.load(self.degraded_files[idx])
        y_arr = np.load(self.gt_files[idx])
        x = torch.from_numpy(x_arr).float().unsqueeze(0)
        y = torch.from_numpy(y_arr).float().unsqueeze(0)
        return x, y
"""
with open('dataset.py', 'w') as f: f.write(dataset_script)
print("Recreated dataset.py for the training pipeline.")

Recreated dataset.py for the training pipeline.


### Step 8: Retraining on TPU (150 Epochs)
Starting the retraining process with 12 RRDB blocks and the hybrid loss function on the TPU.

In [50]:
import os
# Ensure environment is clean of the library conflict
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

# Execute the full 150-epoch training run
!python train.py \
    --train_degraded "/content/extracted_data/datasets/train/train/NoisyLR" \
    --train_gt "/content/extracted_data/datasets/train/train/GT" \
    --checkpoint_dir "./checkpoints" \
    --epochs 150 \
    --batch_size 4 \
    --lr 2e-4

Traceback (most recent call last):
  File "/content/train.py", line 37, in <module>
    if __name__ == '__main__': main()
                               ^^^^^^
  File "/content/train.py", line 16, in main
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 376, in __init__
    sampler = RandomSampler(dataset, generator=generator)  # type: ignore[arg-type]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/sampler.py", line 164, in __init__
    raise ValueError(
ValueError: num_samples should be a positive integer value, but got num_samples=0


### Step 9: Final Evaluation
Performing inference and metric calculation using the final 150-epoch TPU-trained model.

In [51]:
import os
import zipfile

zip_path = '/content/KLA.zip'
extract_path = '/content/extracted_data'

if os.path.exists(zip_path):
    print(f'Extracting {zip_path}...')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print('Extraction successful.')

        print('\n--- Directory Structure Verification ---')
        for root, dirs, files in os.walk(extract_path):
            level = root.replace(extract_path, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 4 * (level + 1)
            for f in files[:2]:
                print(f'{subindent}{f}')
    except Exception as e:
        print(f'Error: {e}')
else:
    print('MISSING DATA: Please upload KLA.zip to /content/')


Extracting /content/KLA.zip...
Extraction successful.

--- Directory Structure Verification ---
extracted_data/
    .venv\Lib\site-packages\torch\include\torch\csrc\api\include\torch\special.h
    datasets\train\train\NoisyLR\000370.npy


In [49]:
train_script = """
import torch
from model import RestorationNet
from losses import CombinedLoss
from dataset import CustomDataset
from torch.utils.data import DataLoader
import os

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RestorationNet(nb=12).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
    criterion = CombinedLoss()

    # Updated paths based on verification output
    base = "/content/extracted_data/datasets/train/train"
    dataset = CustomDataset(os.path.join(base, "NoisyLR"), os.path.join(base, "GT"))
    loader = DataLoader(dataset, batch_size=4, shuffle=True)

    if not os.path.exists('./checkpoints'): os.makedirs('./checkpoints')

    print(f'Starting 150-epoch training on {device}...')
    print(f'Dataset size: {len(dataset)} pairs found.')

    for epoch in range(1, 151):
        model.train()
        epoch_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if epoch % 10 == 0 or epoch == 150:
            torch.save(model.state_dict(), f'./checkpoints/model_epoch_{epoch}.pth')
            print(f'Epoch {epoch}/150, Loss: {epoch_loss/len(loader):.4f}')

if __name__ == '__main__': main()
"""
with open('train.py', 'w') as f: f.write(train_script)
print("train.py updated with correct dataset paths.")

Full train.py script created.


### Step 4: Setup Visualization and Inference Scripts
We need a way to run the model on the test data and then compare the results visually.

In [37]:
inference_script = """import torch
import os
import glob
import numpy as np
from model import RestorationNet
import torch.nn.functional as F

def main(input_dir, output_dir, model_path):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RestorationNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    files = sorted(glob.glob(os.path.join(input_dir, '*.npy')))
    print(f'Running inference on {len(files)} files...')

    with torch.no_grad():
        for f in files:
            name = os.path.basename(f)
            x = torch.from_numpy(np.load(f)).float().unsqueeze(0).unsqueeze(0).to(device)
            pred = model(x)
            # Explicitly upsample to 256x256 for the final output folder
            if pred.shape[-1] != 256:
                pred = F.interpolate(pred, size=(256, 256), mode='bilinear', align_corners=False)
            np.save(os.path.join(output_dir, name), pred.squeeze().cpu().numpy())
    print('Inference complete.')

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--input_dir', type=str)
    parser.add_argument('--output_dir', type=str)
    parser.add_argument('--model_path', type=str)
    args = parser.parse_args()
    main(args.input_dir, args.output_dir, args.model_path)
"""

with open('inference.py', 'w') as f: f.write(inference_script)

visualize_script = """import os
import numpy as np
from PIL import Image

def norm(img):
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)

def main(deg_dir, res_dir, gt_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted([f for f in os.listdir(res_dir) if f.endswith('.npy')])[:10]

    for fname in files:
        deg = np.load(os.path.join(deg_dir, fname))
        res = np.load(os.path.join(res_dir, fname))
        gt = np.load(os.path.join(gt_dir, fname))

        h, w = 256, 256

        def process_to_pil(arr):
            img = Image.fromarray(norm(arr))
            if img.size != (w, h):
                img = img.resize((w, h), Image.BILINEAR)
            return np.array(img)

        canvas = np.zeros((h, w*3), dtype=np.uint8)
        canvas[:, :w] = process_to_pil(deg)
        canvas[:, w:2*w] = process_to_pil(res)
        canvas[:, 2*w:] = process_to_pil(gt)

        Image.fromarray(canvas).save(os.path.join(out_dir, fname.replace('.npy', '.png')))
    print(f'Saved visualizations to {out_dir}')

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--deg_dir', type=str)
    parser.add_argument('--res_dir', type=str)
    parser.add_argument('--gt_dir', type=str)
    parser.add_argument('--out_dir', type=str)
    args = parser.parse_args()
    main(args.deg_dir, args.res_dir, args.gt_dir, args.out_dir)
"""

with open('visualize.py', 'w') as f: f.write(visualize_script)
print("Inference and Visualization scripts updated and corrected.")

Inference and Visualization scripts updated and corrected.


### Step 5: Run Inference and Generate Visualizations
We will now use the latest checkpoint (Epoch 50) to restore images from the test set.

In [38]:
# 1. Run Inference (Now ensuring 256x256 outputs)
!python inference.py \
    --input_dir "/content/extracted_data/datasets/train/train/NoisyLR" \
    --output_dir "/content/restored_results" \
    --model_path "./checkpoints/model_epoch_50.pth"

# 2. Generate Comparison Images
!python visualize.py \
    --deg_dir "/content/extracted_data/datasets/train/train/NoisyLR" \
    --res_dir "/content/restored_results" \
    --gt_dir "/content/extracted_data/datasets/train/train/GT" \
    --out_dir "/content/visualizations"

/content/inference.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))
Traceback (most recent call last):

### Step 6: Display Sample Comparison
Let's view one of the generated comparison images (Degraded | Restored | GT).

In [39]:
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

# Re-scan the directory to find the generated PNGs
vis_files = sorted(glob.glob('/content/visualizations/*.png'))

if vis_files:
    print(f"Found {len(vis_files)} visualization images. Displaying: {os.path.basename(vis_files[0])}")
    img = Image.open(vis_files[0])
    plt.figure(figsize=(20, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Comparison: Degraded (256x256) | Restored (256x256) | Ground Truth (256x256)')
    plt.show()
else:
    print("No visualizations found in /content/visualizations/. Please ensure Step 5 ran successfully.")

No visualizations found in /content/visualizations/. Please ensure Step 5 ran successfully.


### Step 7: Quantitative Evaluation
We will now calculate the average **PSNR** and **SSIM** metrics to evaluate the model's accuracy more rigorously. We will use the `scikit-image` library for these calculations.

In [40]:
evaluate_script = """import os
import numpy as np
import glob
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from PIL import Image

def norm(img):
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)

def main(res_dir, gt_dir):
    res_files = sorted(glob.glob(os.path.join(res_dir, '*.npy')))
    gt_files = sorted(glob.glob(os.path.join(gt_dir, '*.npy')))

    all_psnr = []
    all_ssim = []

    print(f'Evaluating {len(res_files)} files...')

    for r_path, g_path in zip(res_files, gt_files):
        res = np.load(r_path)
        gt = np.load(g_path)

        # Ensure identical normalization for metric calculation
        res_n = norm(res)
        gt_n = norm(gt)

        # Resize res if needed (already handled in inference, but good for safety)
        if res_n.shape != gt_n.shape:
            img = Image.fromarray(res_n).resize((gt_n.shape[1], gt_n.shape[0]), Image.BILINEAR)
            res_n = np.array(img)

        all_psnr.append(psnr(gt_n, res_n, data_range=255))
        all_ssim.append(ssim(gt_n, res_n, data_range=255))

    print(f'--- Final Metrics ---')
    print(f'Average PSNR: {np.mean(all_psnr):.4f} dB')
    print(f'Average SSIM: {np.mean(all_ssim):.4f}')

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--res_dir", type=str)
    parser.add_argument("--gt_dir", type=str)
    args = parser.parse_args()
    main(args.res_dir, args.gt_dir)
"""

with open('evaluate_metrics.py', 'w') as f: f.write(evaluate_script)
print("Evaluation script created.")

Evaluation script created.


In [41]:
# Run the quantitative evaluation
!python evaluate_metrics.py \
    --res_dir "/content/restored_results" \
    --gt_dir "/content/extracted_data/datasets/train/train/GT"

Evaluating 0 files...
--- Final Metrics ---
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
Average PSNR: nan dB
Average SSIM: nan


In [42]:
!python train.py \
    --train_degraded "/content/extracted_data/datasets/train/train/NoisyLR" \
    --train_gt "/content/extracted_data/datasets/train/train/GT" \
    --checkpoint_dir "./checkpoints" \
    --epochs 150 \
    --batch_size 4 \
    --lr 2e-4

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


### Step 9 (Retry): Final Inference and Evaluation
Executing inference and metric calculation using the 150-epoch checkpoint.

In [43]:
# Run Inference
!python inference.py \
    --input_dir "/content/extracted_data/datasets/train/train/NoisyLR" \
    --output_dir "/content/restored_results_final" \
    --model_path "./checkpoints/model_epoch_150.pth"

# Run Metrics
!python evaluate_metrics.py \
    --res_dir "/content/restored_results_final" \
    --gt_dir "/content/extracted_data/datasets/train/train/GT"

/content/inference.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))
Traceback (most recent call last):

### Step 9: Post-Retraining Evaluation
We verify the performance of the enhanced model using the final checkpoint.